# Computing time, memory, and computational cost

This notebook measures *observed wall-clock time on the executing machine*. A full spectral factorization is not universally faster: it incurs O(n³) preprocessing and O(n²) memory, whereas a banded solve can be efficient for one fit. Results are not transferable across machines.

## Machine metadata

Record system, CPU, Python, SciPy, NumPy, and thread-control variables. Results depend on BLAS, thermals, power mode, and concurrent programs.

In [ ]:
import os,platform,sys,time
from math import comb
import numpy as np,scipy
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded
from trendestimation import TrendSmoother
for name,value in {"platform":platform.platform(),"processor":platform.processor(),"architecture":platform.machine(),"logical_cpus":os.cpu_count(),"python":sys.version.split()[0],"numpy":np.__version__,"scipy":scipy.__version__,"OPENBLAS_NUM_THREADS":os.environ.get("OPENBLAS_NUM_THREADS","unset"),"MKL_NUM_THREADS":os.environ.get("MKL_NUM_THREADS","unset")}.items():
    print(name,":",value)

## Build the banded reference directly

For d fixed, Q=D_dᵀD_d is banded with half-bandwidth d. Here we accumulate each local outer product directly into the banded representation (no dense n×n Q) and solve (I+λQ)τ=y for μ=0.

In [ ]:
def banded_penalty(n,d):
    coeff=[(-1)**(d-j)*comb(d,j) for j in range(d+1)]
    ab=np.zeros((2*d+1,n))
    for r in range(n-d):
        for a,ca in enumerate(coeff):
            i=r+a
            for b,cb in enumerate(coeff):
                j=r+b
                ab[d+i-j,j]+=ca*cb
    return ab

def solve_trend_banded(y,bands,lam,d):
    ab=lam*bands.copy()
    ab[d]+=1
    return solve_banded((d,d),ab,y,overwrite_ab=True,check_finite=False)

def median_time(fn,reps=3):
    fn()
    times=[]
    for _ in range(reps):
        s=time.perf_counter();fn();times.append(time.perf_counter()-s)
    return float(np.median(times))

## Compare repeated smoothing

Compare total setup plus repeated fits, and validate parity against the structured solver. Small defaults keep documentation builds reasonable; manually use larger sizes for a serious benchmark.

In [ ]:
sizes=[30,50,75] if os.environ.get("TRENDESTIMATION_FAST")=="1" else [40,70,100]
results=[]
for n in sizes:
    d=2;y=np.random.default_rng(120+n).normal(size=n)
    lambdas=np.geomspace(.001,1000,12)
    setup_s=median_time(lambda:TrendSmoother(n,d))
    sm=TrendSmoother(n,d)
    setup_b=median_time(lambda:banded_penalty(n,d))
    bands=banded_penalty(n,d)
    time_s=median_time(lambda:[sm.fit(y,lambda_=float(lam)).trend for lam in lambdas])
    time_b=median_time(lambda:[solve_trend_banded(y,bands,float(lam),d) for lam in lambdas])
    err=max(np.max(np.abs(sm.fit(y,lambda_=float(lam)).trend-solve_trend_banded(y,bands,float(lam),d))) for lam in lambdas)
    results.append((n,setup_s,time_s,setup_b,time_b,float(err)))
    print(f"n={n}: spectral={setup_s+time_s:.4g}s; banded={setup_b+time_b:.4g}s; max difference={err:.2e}")
assert max(r[-1] for r in results)<1e-6

## Visualize runtime and theoretical storage

The 8n²-byte line is *analytical eigenvector storage*, not measured memory. Observe setup time separately from the 12 requested fits.

In [ ]:
r=np.asarray(results)
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].plot(r[:,0],r[:,1]+r[:,2],"o-",label="SVD setup + 12 fits")
axs[0].plot(r[:,0],r[:,3]+r[:,4],"s-",label="Banded setup + 12 fits")
axs[0].set(xlabel="n",ylabel="Seconds",title="Measured computation time");axs[0].legend()
axs[1].plot(r[:,0],8*r[:,0]**2/1024**2,"o-")
axs[1].set(xlabel="n",ylabel="MiB",title="One stored float64 n×n basis")
fig.tight_layout();plt.show()

## Optional MXN electricity estimates

Energy ≈ watts × elapsed seconds / (1000 × 3600) kWh; cost = energy × tariff in MXN/kWh. Neither value is an actual measurement until you provide independently measured power consumption.

In [ ]:
assumed_system_watts=None   # edit on your own machine
tariff_mxn_per_kwh=None      # use a documented, actual electricity tariff
elapsed=sum(float(row[1]+row[2]) for row in results)
if assumed_system_watts is None or tariff_mxn_per_kwh is None:
    print("Energy/electricity cost not estimated: supply power and tariff.")
else:
    energy=assumed_system_watts*elapsed/(1000*3600)
    print(f"Assumed energy {energy:.7g} kWh; electricity {energy*tariff_mxn_per_kwh:.7g} MXN")